[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuchnick/intro-to-ml-tutoring/blob/main/session01/03_numpy.ipynb)

# 03 · numpy: fast maths on whole arrays

A Python list can hold anything, which makes it flexible but slow for maths. A **numpy array** holds numbers of a single type packed together in memory, so you can do maths on **all of them at once**, with no loop needed. Every ML library is built on top of numpy, including scikit-learn, which you've already used in the recitations. Now you'll see what it does underneath.

Fell behind? Run `git pull --no-edit` and open `session01/solutions/03_numpy.ipynb`.

In [1]:
# ✍️ Import numpy. Everyone abbreviates it to np.
import numpy as np

## 1. Why numpy? Speed.

In [2]:
# ✍️ Double a million numbers, first with a loop and then with numpy
import time

numbers = list(range(1_000_000))
start = time.perf_counter()
doubled = [x * 2 for x in numbers]
t_list = time.perf_counter() - start

arr = np.arange(1_000_000)
start = time.perf_counter()
doubled_arr = arr * 2
t_np = time.perf_counter() - start

print(f"list: {t_list*1000:.1f} ms | numpy: {t_np*1000:.1f} ms | numpy is ~{t_list/t_np:.0f}x faster")

list: 13.9 ms | numpy: 1.7 ms | numpy is ~8x faster


## 2. Arrays have a shape

In ML, the data is a **2D array** usually called `X`. It has **one row per sample** (patient) and **one column per feature** (measurement).

In [3]:
# ✍️ 4 patients × 2 features (radius_mean, texture_mean), taken from our dataset
X = np.array([
    [17.99, 10.38],
    [20.57, 17.77],
    [11.42, 20.38],
    [12.45, 15.70],
])
X.shape, X.ndim, X.dtype

((4, 2), 2, dtype('float64'))

In [4]:
# ✍️ Indexing works as [row, column]. A `:` on its own means "all of them".
X[0]        # first patient (a row)

array([17.99, 10.38])

In [5]:
X[:, 0]     # the first feature for every patient (a column)

array([17.99, 20.57, 11.42, 12.45])

In [6]:
X[1, 1]     # patient 1, feature 1

np.float64(17.77)

In [7]:
# 🤔 Predict: what are the shapes of these three?
print(X[2].shape, X[:, 1].shape, X[1:3].shape)

(2,) (4,) (2, 2)


## 3. Maths on the whole array at once (vectorisation)

In [8]:
# ✍️ No loop needed: the operation is applied to every element
X * 10

array([[179.9, 103.8],
       [205.7, 177.7],
       [114.2, 203.8],
       [124.5, 157. ]])

In [9]:
# ✍️ axis=0 squashes the rows, giving one result per column (feature)
X.mean(axis=0)

array([15.6075, 16.0575])

In [10]:
# 🤔 Predict: what is the shape of X.mean(axis=1), and what does each number in it mean?
X.mean(axis=1)

array([14.185, 19.17 , 15.9  , 14.075])

## 4. Boolean masks: selecting rows with a condition

This is the way you'll filter data from now on.

In [11]:
# ✍️ Comparing an array to a number gives an array of True/False
radius = X[:, 0]
radius > 15

array([ True,  True, False, False])

In [12]:
# ✍️ Use that True/False array to keep only the matching rows
X[radius > 15]

array([[17.99, 10.38],
       [20.57, 17.77]])

## 5. Broadcasting: one row minus every row

What happens if you subtract a single patient (shape `(2,)`) from `X` (shape `(4, 2)`)? numpy **stretches** the row so that it's subtracted from **every** row. This is called broadcasting.

In [13]:
# ✍️ Distance from a new patient to every patient in X, in one line, with no loop
new_patient = np.array([13.0, 16.0])
diff = X - new_patient
distances = np.sqrt((diff ** 2).sum(axis=1))
distances

array([7.51561707, 7.7741752 , 4.6562646 , 0.6264982 ])

In [14]:
# ✍️ argsort gives the indices that would sort the array, so the nearest patients come first
np.argsort(distances)

array([3, 2, 0, 1])

`np.argsort(distances)[:k]` gives the **k nearest neighbours**. That is almost exactly the line from the kNN recitation. Compare it with your loop-based `euclidean_distance` from notebook 02.

## 🏋️ Exercise 1

1. Write `euclidean_distance_np(a, b)` using numpy, **without a loop**, in a single line.
2. Using `X` and `new_patient` from above, store the index of the **2 nearest** patients in `nearest_2`, nearest first.
3. Store the **mean texture** (the column at index 1) of the patients whose radius is **below 15** in `mean_texture_small`.

In [15]:
def euclidean_distance_np(a, b):
    return np.sqrt(np.sum((np.array(a) - np.array(b)) ** 2))

nearest_2 = np.argsort(distances)[:2]
mean_texture_small = X[X[:, 0] < 15, 1].mean()

In [16]:
# --- Check (do not edit) ---
assert nearest_2 is not None and mean_texture_small is not None, "Some variables are still None."
assert euclidean_distance_np([0, 0], [3, 4]) == 5, "euclidean_distance_np([0, 0], [3, 4]) should be 5. Did you `return`?"
assert list(nearest_2) == [3, 2], "the nearest two should be patients 3 and 2. Did you use argsort and [:2]?"
assert round(mean_texture_small, 2) == 18.04, "filter the rows with radius < 15 first, then take column 1"
print("✅ Exercise 1 correct!")

✅ Exercise 1 correct!


## 🔄 Checkpoint

Save the notebook (**Ctrl+S**, or **Cmd+S** on Mac).

**Fell behind?** In the terminal, run `git pull --no-edit`, then open `session01/solutions/03_numpy.ipynb`.